In [1]:
from delta import *
from pyspark.sql import *
from pyspark.sql.functions import *

get_ipython().run_line_magic('load_ext', 'sparksql_magic')
get_ipython().run_line_magic('config', 'SparkSql.limit=20')

spark = (SparkSession.builder
           .appName("streaming-basics")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .getOrCreate())

# spark.sparkContext.setLogLevel("INFO")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/01 08:35:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
lines = (spark.readStream
         .format("socket")
         .option("host", "localhost")
         .option("port", 9999)
         .load())

25/06/01 08:35:34 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.


In [3]:
words = lines.select(
    explode(split(lines.value, " ")).alias("word"))

In [4]:
wordCounts = words.groupBy("word").count()

In [5]:
import socket

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    port_in_use = s.connect_ex(('localhost', 9999)) == 0

port_in_use

True

In [6]:
query = (wordCounts.writeStream
         .outputMode("complete")
         .format("console")
         .start())

25/06/01 08:35:35 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-6ae5fd09-272a-4d9e-bc09-f1808e5fecda. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/06/01 08:35:36 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [9]:
query.stop()

In [10]:
spark.stop()